<center>
<small>
Nova Information Management School<br>
Bachelor in Data Science, 2025-26<br>
Machine Learning I<br><br>
Project developed by Group 7<br>
Carolina Arez 20241739 · Henrique Madureira 20241756 · Miguel Melo 20241757 · Noa Penas 20241682
</small>
</center>

<hr style="border: 0.5px solid #E89C63;">

# <font color='#E89C63'>Notebook 9: Final model - Kaggle </font> <a class="anchor" id="P0"></a>
<div style="font-size:15px; line-height:1.5; margin-bottom:10px;">
In this notebook, we focus on <strong>generating final predictions for the Kaggle submission</strong>.
<br><br>
Here, we <strong>reapply all transformations, feature engineering, feature selection, and hyperparameter tuning (HPT)</strong> from Notebooks 2,3 and 4, and apply them to the <strong><code>predict</code> dataset</strong>. The main steps in this notebook are:
<br><br>
<strong>1.</strong> Reapply all <strong>transformations and preprocessing</strong> in <code>learn</code>;<br>
<strong>2.</strong> Apply <strong>feature selection (FS)</strong>;<br>
<strong>3.</strong> Apply the <strong>final model</strong> with the <strong>optimized hyperparameters</strong>;<br>
<strong>4.</strong> Reapply all transformations and the model to the <strong><code>predict</code> dataset</strong>;<br>
<strong>5.</strong> Generate the <strong>final predictions</strong>;<br>
<strong>6.</strong> Create and save the <strong>submission file</strong> (<code>submission_GO7.csv</code>) ready for Kaggle.
<br><br>
This notebook is focused on <strong>inference and submission preparation</strong>.
</div>


### <font color='#E89C63'>Libraries </font> <a class="anchor" id="P1"></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, RobustScaler, MinMaxScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer

<br>

### <font color='#E89C63'>Final Submission Code </font> <a class="anchor" id="P1"></a>
<div style="font-size:15px; line-height:1.5; margin-bottom:10px;">
Following the optimal model, it was <strong>trained on the complete <code>learn</code> dataset</strong> to fully utilize the available information, and subsequently applied to generate predictions on the <code>predict</code> dataset.
</div>

In [2]:
#0.791 score on Kaggle 

# -----------------------------------------------------
# Function and class for preprocessing
# -----------------------------------------------------

def iqr_clip_num(df, num_cols, Q1, Q3):
    df_c = df.copy()
    for col in num_cols:
        iqr = Q3[col] - Q1[col]
        lower = Q1[col] - 1.5 * iqr
        upper = Q3[col] + 1.5 * iqr
        df_c[col] = df_c[col].clip(lower, upper)
    return df_c[num_cols]

class LogSkewedTransformer:
    def __init__(self, threshold=1.0):
        self.threshold = threshold
        self.skewed_cols = []
        self.shift_vals = {}

    def fit(self, df, num_cols):
        skew_vals = df[num_cols].skew()
        self.skewed_cols = skew_vals[skew_vals > self.threshold].index.tolist()
        for col in self.skewed_cols:
            min_val = df[col].min()
            self.shift_vals[col] = -min_val + 1 if min_val <= 0 else 0

    def transform(self, df):
        df_c = df.copy()
        for col in self.skewed_cols:
            df_c[col] = np.log1p(df_c[col] + self.shift_vals[col])
        return df_c


# -----------------------------------------------------
# Data Loading, Cleaning, and Feature Engineering
# -----------------------------------------------------

categorical_cols = ['origin_Lisboa', 'origin_Porto']
learn = pd.read_csv('Nata_Files/learn.csv', index_col=0)
learn = learn.drop(columns=['notes_baker', 'pastry_type'], errors='ignore')
learn = learn.drop([1, 2917], errors='ignore').reset_index(drop=True)
learn = learn.dropna(subset=['quality_class'])
learn['origin'] = learn['origin'].str.strip().str.title()
learn['salt_ratio'] = learn['salt_ratio'] / 10

learn['origin_Lisboa'] = (learn['origin'] == 'Lisboa').astype(int)
learn['origin_Porto']  = (learn['origin'] == 'Porto').astype(int)
learn = learn.drop(columns=['origin'], errors='ignore')

learn['cream_x_egg'] = learn['cream_fat_content'] * learn['egg_yolk_count']
learn['Egg_Yolk_Rate_Cooking'] = learn['egg_yolk_count'] / learn['baking_duration']
learn['sugar_per_egg'] = learn['sugar_content'] / learn['egg_yolk_count']
learn['salt_per_sugar'] = learn['salt_ratio'] / learn['sugar_content']
# removed X['temp_diff'] = X['oven_temperature'] - X['final_temperature']
learn['total_stress'] = learn['baking_duration'] * learn['oven_temperature']
# removed X['oven_humidity_interaction'] = X['oven_temperature'] * X['ambient_humidity']
learn['is_burnt'] = (learn['baking_duration'] > 40).astype(int)
categorical_cols.append('is_burnt')
mask_sugar_high = learn['sugar_content'] > 100
learn.loc[mask_sugar_high, 'sugar_content'] = learn.loc[mask_sugar_high, 'sugar_content'] / 10

learn = learn.drop(columns = (['ambient_humidity', 'lemon_zest_ph']))


X = learn.drop(columns=['quality_class'])
y = learn['quality_class']

categorical_cols = [c for c in X.columns if c.startswith('origin_')]
numerical_cols = [c for c in X.columns if c not in categorical_cols]


# -----------------------------------------------------
# Preprocessing
# -----------------------------------------------------

Q1 = X[numerical_cols].quantile(0.25)
Q3 = X[numerical_cols].quantile(0.75)
X[numerical_cols] = iqr_clip_num(X, numerical_cols, Q1, Q3)

imputer_final = SimpleImputer(strategy='median')
X[numerical_cols] = imputer_final.fit_transform(X[numerical_cols])

log_transformer = LogSkewedTransformer(threshold=1.0)
log_transformer.fit(X, numerical_cols)
X = log_transformer.transform(X)

scaler = RobustScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

# Label encode target
le_y = LabelEncoder()
y_encoded = le_y.fit_transform(y)


# -----------------------------------------------------
# Final Model
# -----------------------------------------------------

gb_params = {
    'learning_rate': 0.05,
    'max_depth': 7,
    'min_samples_split': 5,
    'n_estimators': 300,
    'subsample': 0.7,
    'random_state': 42
}


model = GradientBoostingClassifier(**gb_params)
model.fit(X, y_encoded)

# ------------------------------------------------------------
# Data Loading, Cleaning, Feature Engineering and Preprocessing
# ------------------------------------------------------------

predict = pd.read_csv('Nata_Files/predict.csv', index_col=0)
predict_index = predict.index
predict = predict.drop(columns=['notes_baker', 'pastry_type'], errors='ignore')
predict['origin'] = predict['origin'].str.strip().str.title()
predict['salt_ratio'] = predict['salt_ratio'] / 10
predict[['baking_duration', 'cooling_period']] = predict[['baking_duration', 'cooling_period']].astype(float)
predict['origin_Lisboa'] = (predict['origin'] == 'Lisboa').astype(int)
predict['origin_Porto']  = (predict['origin'] == 'Porto').astype(int)
predict = predict.drop(columns=['origin'], errors='ignore')

predict['cream_x_egg'] = predict['cream_fat_content'] * predict['egg_yolk_count']
predict['Egg_Yolk_Rate_Cooking'] = predict['egg_yolk_count'] / predict['baking_duration']
predict['sugar_per_egg'] = predict['sugar_content'] / predict['egg_yolk_count']
predict['salt_per_sugar'] = predict['salt_ratio'] / predict['sugar_content']
# removed X['temp_diff'] = X['oven_temperature'] - X['final_temperature']
predict['total_stress'] = predict['baking_duration'] * predict['oven_temperature']
# removed X['oven_humidity_interaction'] = X['oven_temperature'] * X['ambient_humidity']
predict['is_burnt'] = (predict['baking_duration'] > 40).astype(int)
mask_sugar_high = predict['sugar_content'] > 100
predict.loc[mask_sugar_high, 'sugar_content'] = predict.loc[mask_sugar_high, 'sugar_content'] / 10
X_predict = predict[X.columns].copy()

X_predict[numerical_cols] = iqr_clip_num(X_predict, numerical_cols, Q1, Q3)
X_predict[numerical_cols] = imputer_final.transform(X_predict[numerical_cols])
X_predict = log_transformer.transform(X_predict)
X_predict[numerical_cols] = scaler.transform(X_predict[numerical_cols])

# ------------------------------------------------------------
# Submission
# ------------------------------------------------------------

preds_enc = model.predict(X_predict)
preds_labels = le_y.inverse_transform(preds_enc)

submission = pd.DataFrame({
    'id': predict_index,
    'quality_class': preds_labels
})
submission.to_csv('Nata_Files/submission_GO7.csv', index=False)

print("Predictions saved to 'Nata_Files/submission_GO7.csv'")

Predictions saved to 'Nata_Files/submission_GO7.csv'
